In [3]:
!pip install spacy xgboost scikit-learn pandas numpy joblib fastapi uvicorn nest-asyncio openai python-dotenv

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.1 MB 991.0 kB/s eta 0:00:02
   -- ------------------------------------- 0.1/1.1 MB 1.1 MB/s eta 0:00:01
   -- ------------------------------------- 0.1/1.1 MB 1.1 MB/s eta 0:00:01
   ------ --------------------------------- 0.2/1.1 MB 876.1 kB/s eta 0:00:02
   ------ --------------------------------- 0.2/1.1 MB 857.5 kB/s eta 0:00:02
   ------ --------------------------------- 0.2/1.1 MB 857.5 kB/s eta 0:00:02
   ---------- ----------------------------- 0.3/1.1 MB 905.4 kB/s eta 0:00:01
   ---------- ----------------------------- 0.3/1.1 MB 905.4 kB/s eta 0:00:01
   ------------ --------------------------- 0.3/1.1 MB 832.3 kB/s eta 0:00:01
   ------------------- -------------------- 0.6/1.1 MB 1.2 MB/s eta 0:00:01
   ------------------------ --------------- 0.7/1.1 MB 1.5 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.1 MB 1.4 MB/s eta 0:00:01
   --

In [4]:
!python -m spacy download ru_core_news_sm

     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     --------------------------------------- 0.1/15.3 MB 656.4 kB/s eta 0:00:24
     --------------------------------------- 0.1/15.3 MB 939.4 kB/s eta 0:00:17
     --------------------------------------- 0.1/15.3 MB 774.0 kB/s eta 0:00:20
      --------------------------------------- 0.2/15.3 MB 1.2 MB/s eta 0:00:13
      --------------------------------------- 0.3/15.3 MB 1.2 MB/s eta 0:00:13
      --------------------------------------- 0.4/15.3 MB 1.1 MB/s eta 0:00:14
     - -------------------------------------- 0.4/15.3 MB 1.1 MB/s eta 0:00:15
     - -------------------------------------- 0.4/15.3 MB 1.1 MB/s eta 0:00:15
     - -------------------------------------- 0.6/15.3 MB 1.3 MB/s eta 0:00:12
     - -------------------------------------- 0.6/15.3 MB 1.3 MB/s eta 0:00:12
     - -------------------------------------- 0.7/15.3 MB 1.3 M

In [5]:
import spacy
import re

nlp = spacy.load("ru_core_news_sm")

LEADERSHIP_SIGNALS = {
    "vision": ["цель", "будущее", "стратегия", "видение", "направление", "развитие", "план", "перспектива", "миссия", "приоритет"],
    "responsibility": ["ответственность", "решение", "взял", "принял", "обязанность", "результат", "выполнил", "достиг", "обеспечил", "отвечал"],
    "team": ["команда", "коллеги", "вместе", "помог", "поддержал", "мотивировал", "делегировал", "доверил", "координировал", "объединил"],
    "initiative": ["предложил", "инициировал", "запустил", "создал", "разработал", "первым", "сам", "самостоятельно", "придумал", "внедрил"],
    "resilience": ["преодолел", "несмотря", "трудность", "препятствие", "справился", "продолжил", "не сдался", "вызов", "кризис", "решил"],
    "influence": ["убедил", "вдохновил", "повлиял", "изменил", "показал", "научил", "передал", "объяснил", "донёс"]
}

def extract_features(text: str) -> dict:
    doc = nlp(text.lower())
    words = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    
    features = {}
    for competency, keywords in LEADERSHIP_SIGNALS.items():
        hits = sum(1 for w in words if any(kw in w for kw in keywords))
        features[f"{competency}_score"] = min(hits / max(len(words) * 0.05, 1), 1.0)
    
    first_person = len(re.findall(r'\b(я|мне|мой|моя|моё|мои|меня|мною)\b', text.lower()))
    features["first_person_ratio"] = first_person / max(len(text.split()), 1)
    
    verbs = [t for t in doc if t.pos_ == "VERB"]
    features["action_verb_ratio"] = len(verbs) / max(len(words), 1)
    
    features["text_length"] = len(text.split())
    features["sentence_count"] = len(list(doc.sents))
    features["concreteness"] = min(len(re.findall(r'\d+', text)) / 5, 1.0)
    
    return features

print("features.py готов ✓")

features.py готов ✓


In [6]:
def calculate_scores(features: dict) -> dict:
    competencies = ["vision", "responsibility", "team", "initiative", "resilience", "influence"]
    
    scores = {}
    for comp in competencies:
        base = features.get(f"{comp}_score", 0)
        concreteness_boost = features.get("concreteness", 0) * 0.15
        action_boost = features.get("action_verb_ratio", 0) * 0.2
        first_person_boost = min(features.get("first_person_ratio", 0) * 2, 0.15)
        
        raw = base + concreteness_boost + action_boost + first_person_boost
        scores[comp] = round(min(raw * 10, 10), 1)
    
    length = features.get("text_length", 0)
    if length < 100:
        penalty = (100 - length) / 100 * 0.4
        scores = {k: round(max(v - v * penalty, 0), 1) for k, v in scores.items()}
    
    scores["overall"] = round(sum(scores[c] for c in competencies) / len(competencies), 1)
    return scores

print("scorer.py готов ✓")


scorer.py готов ✓


In [7]:
!pip install sentence-transformers

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ----------------------------- ---------- 30.7/41.5 kB 1.4 MB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 664.6 kB/s eta 0:00:00
   ---------------------------------------- 0.0/512.4 kB ? eta -:--:--
   --- ----------------------------------- 41.0/512.4 kB 960.0 kB/s eta 0:00:01
   ------ --------------------------------- 81.9/512.4 kB 1.5 MB/s eta 0:00:01
   ------- ------------------------------- 92.2/512.4 kB 744.7 kB/s eta 0:00:01
   --------------- ------------------------ 204.8/512.4 kB 1.1 MB/s eta 0:00:01
   --------------- ------------------------ 204.8/512.4 kB 1.1 MB/s eta 0:00:01
   ---------------- --------------------- 225.3/512.4 kB 860.2 kB/s eta 0:00:01
   -------------------------- ------------- 337.9/512.4 kB 1.0 MB/s eta 0:00:01
   -------------------------- ------------- 337.9/512.4 kB 1.0 MB/s eta 0:00:01
   -------------------------- ----------- 358.4/512.4 kB 8

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Скачает multilingual BERT (~90MB), один раз
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def get_embedding(text: str) -> np.ndarray:
    return embedder.encode(text)

print("BERT embedder готов ✓")
print(f"Размер вектора: {get_embedding('тест').shape}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\Acer\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BERT embedder готов ✓
Размер вектора: (384,)


In [54]:
!pip install deep-translator

   ---------------------------------------- 0.0/42.3 kB ? eta -:--:--
   ----------------------------- ---------- 30.7/42.3 kB 660.6 kB/s eta 0:00:01
   ---------------------------------------- 42.3/42.3 kB 681.8 kB/s eta 0:00:00


In [55]:
from deep_translator import GoogleTranslator

def translate_to_english(text: str) -> str:
    # Разбиваем на куски по 4500 символов (лимит Google Translate)
    chunks = [text[i:i+4500] for i in range(0, len(text), 4500)]
    translated = [GoogleTranslator(source='ru', target='en').translate(chunk) for chunk in chunks]
    return " ".join(translated)

# Тест
print(translate_to_english("Я взял на себя ответственность за результат команды"))

I took responsibility for the team's results


In [25]:
!pip install sentencepiece tiktoken

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.1 MB 660.6 kB/s eta 0:00:02
   - -------------------------------------- 0.0/1.1 MB 281.8 kB/s eta 0:00:04
   -- ------------------------------------- 0.1/1.1 MB 393.8 kB/s eta 0:00:03
   ---- ----------------------------------- 0.1/1.1 MB 504.4 kB/s eta 0:00:02
   ---- ----------------------------------- 0.1/1.1 MB 514.3 kB/s eta 0:00:02
   -------- ------------------------------- 0.2/1.1 MB 724.0 kB/s eta 0:00:02
   -------- ------------------------------- 0.2/1.1 MB 654.9 kB/s eta 0:00:02
   ---------- ----------------------------- 0.3/1.1 MB 680.9 kB/s eta 0:00:02
   ------------- -------------------------- 0.4/1.1 MB 825.0 kB/s eta 0:00:01
   --------------- ------------------------ 0.4/1.1 MB 804.0 kB/s eta 0:00:01
   ---------------- ----------------------- 0.4/1.1 MB 860.2 kB/s eta 0:00:01
   -

In [27]:
from transformers import pipeline

print("Загружаю модель...")

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    use_fast=False
)

print("Модель загружена ✓")

Загружаю модель...


config.json: 0.00B [00:00, ?B/s]

C:\Users\Acer\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Модель загружена ✓


In [56]:
def detect_concreteness(text: str) -> float:
    doc = nlp(text.lower())
    score = 0
    
    # 1. Числа и факты
    numbers = re.findall(r'\d+', text)
    score += min(len(numbers) * 0.1, 0.3)
    
    # 2. Маркеры реальных историй под S-LPI практики
    story_markers = [
        # Общие маркеры конкретного опыта
        "например", "однажды", "когда я", "в тот момент",
        "во время", "в ходе", "после того как", "в результате",
        "мне удалось", "я добился", "я столкнулся",
        
        # Model the Way — личный пример и ценности
        "я придерживаюсь", "для меня важно", "я всегда",
        "мои принципы", "я верю что", "я показал на примере",
        
        # Inspire a Shared Vision — видение и вовлечение
        "я предложил идею", "я убедил команду", "мы вместе решили",
        "я объяснил цель", "люди поддержали", "я вдохновил",
        
        # Challenge the Process — инновации и риск
        "я предложил новый подход", "мы изменили процесс",
        "я рискнул", "никто раньше не делал", "я попробовал",
        "я заметил проблему и", "вместо того чтобы",
        
        # Enable Others to Act — развитие других
        "я делегировал", "я доверил", "я помог им",
        "благодаря моей поддержке", "я создал условия",
        "каждый мог", "я развил в них",
        
        # Encourage the Heart — признание вклада
        "я поблагодарил", "я отметил вклад", "мы отпраздновали",
        "я признал", "я поддержал морально", "команда почувствовала"
    ]
    
    hits = sum(1 for m in story_markers if m in text.lower())
    score += min(hits * 0.12, 0.4)
    
    # 3. Глаголы прошедшего времени = реальный опыт, не намерения
    past_verbs = [t for t in doc if t.pos_ == "VERB" and "Past" in t.morph.get("Tense", [])]
    future_verbs = [t for t in doc if t.pos_ == "VERB" and "Fut" in t.morph.get("Tense", [])]
    
    if len(past_verbs) > len(future_verbs):
        score += 0.3   # реальный опыт
    else:
        score -= 0.15  # обещания и намерения — штраф
    
    return round(max(0.4, min(1.0, score)), 2)

print("Story markers под S-LPI обновлены ✓")

def calculate_scores_zeroshot(text: str) -> dict:
    
    competency_pairs = {
        "model_the_way": (
            "автор приводит конкретные примеры где он лично демонстрировал свои ценности и был примером для других",
            "автор только говорит о своих ценностях без конкретных примеров поведения"
        ),
        "inspire_shared_vision": (
            "автор конкретно описывает как он вовлекал других людей в общую цель и вдохновлял команду",
            "автор пишет только о личных амбициях не упоминая влияние на других людей"
        ),
        "challenge_the_process": (
            "автор описывает конкретный случай где он предложил новый подход или изменил существующий процесс",
            "автор следует стандартным путём и не описывает случаев где менял что-то или рисковал"
        ),
        "enable_others_to_act": (
            "автор конкретно описывает как помогал другим людям действовать развиваться и вносить вклад",
            "автор не упоминает других людей и их развитие фокусируясь только на своих достижениях"
        ),
        "encourage_the_heart": (
            "автор описывает конкретные случаи где признавал вклад других и поддерживал людей",
            "автор не упоминает признание других людей и пишет исключительно про себя"
        )
    }
    
    concreteness = detect_concreteness(text)
    practices = list(competency_pairs.keys())
    scores = {}
    
    for comp, (positive, negative) in competency_pairs.items():
        result = classifier(
            text,
            candidate_labels=[positive, negative],
            hypothesis_template="В этом тексте: {}"
        )
        pos_idx = result["labels"].index(positive)
        pos_score = result["scores"][pos_idx]
        neg_score = result["scores"][1 - pos_idx]
        
        # Используем логарифмическое соотношение — лучше различает крайние значения
        import math
        ratio = pos_score / max(neg_score, 0.001)
        log_score = math.log(ratio)  # от -7 до +7 примерно
        
        # Нормализуем в 0-10
        # log(0.71/0.29) = 0.89  → слабый encourage
        # log(0.99/0.01) = 4.6   → сильный
        normalized = (log_score) / 5.0  # делим на 5 чтобы получить 0-1 диапазон
        normalized = max(0, min(1, normalized))
        scores[comp] = round(normalized * 10, 1)
    
    avg = sum(scores[p] for p in practices) / len(practices)
    scores["overall"] = round(avg * concreteness, 1)
    scores["concreteness"] = concreteness
    
    return scores

print("Логарифмическая нормализация ✓")

def generate_rule_based_feedback(scores: dict) -> dict:
    practice_names = {
        "model_the_way": "Model the Way",
        "inspire_shared_vision": "Inspire a Shared Vision",
        "challenge_the_process": "Challenge the Process",
        "enable_others_to_act": "Enable Others to Act",
        "encourage_the_heart": "Encourage the Heart"
    }
    
    tips = {
        "model_the_way": (
            "Хорошо показаны личные ценности и пример для других",
            "Покажи как твои действия отражают твои принципы"
        ),
        "inspire_shared_vision": (
            "Чётко сформулировано видение и общая цель",
            "Опиши какое будущее ты хочешь создать и как вовлекаешь других"
        ),
        "challenge_the_process": (
            "Виден инновационный подход и готовность к риску",
            "Приведи пример где ты предложил новый подход или изменил процесс"
        ),
        "enable_others_to_act": (
            "Хорошо показано как ты развиваешь и поддерживаешь других",
            "Расскажи как ты помогал другим расти и действовать самостоятельно"
        ),
        "encourage_the_heart": (
            "Виден искренний интерес к людям и их достижениям",
            "Добавь пример где ты признавал вклад других или поддерживал команду"
        )
    }
    
    strengths = []
    growth_areas = []
    feedback_by_practice = {}
    
    for practice, name in practice_names.items():
        score = scores.get(practice, 0)
        positive, negative = tips[practice]
        
        if score >= 6:
            strengths.append(name)
            feedback_by_practice[practice] = f"✓ {positive} ({score}/10)"
        else:
            growth_areas.append(name)
            feedback_by_practice[practice] = f"→ {negative} ({score}/10)"
    
    overall = scores.get("overall", 0)
    if overall >= 7:
        leader_type = "Exemplary Leader (S-LPI)"
    elif overall >= 5:
        leader_type = "Developing Leader (S-LPI)"
    else:
        leader_type = "Emerging Leader (S-LPI)"
    
    return {
        "leader_type": leader_type,
        "framework": "Kouzes & Posner — Student Leadership Practices Inventory",
        "strengths": strengths,
        "growth_areas": growth_areas,
        "by_practice": feedback_by_practice
    }

print("S-LPI фреймворк (Kouzes & Posner) готов ✓")

Story markers под S-LPI обновлены ✓
Английский zero-shot готов ✓
S-LPI фреймворк (Kouzes & Posner) готов ✓


In [60]:
# Диагностика — что находит и что не находит
for name, essay in [("СИЛЬНОЕ", essay_strong), ("СЛАБОЕ", essay_weak), ("ВОЗДУХ", essay_air)]:
    doc = nlp(essay.lower())
    numbers = re.findall(r'\d+', essay)
    past_verbs = [t.text for t in doc if t.pos_ == "VERB" and "Past" in t.morph.get("Tense", [])]
    future_verbs = [t.text for t in doc if t.pos_ == "VERB" and "Fut" in t.morph.get("Tense", [])]
    
    story_markers = [
        "например", "однажды", "когда я", "в тот момент",
        "во время", "в ходе", "после того как", "в результате",
        "мне удалось", "я добился", "я столкнулся",
        "я предложил идею", "я убедил команду", "мы вместе решили",
        "я объяснил цель", "люди поддержали", "я вдохновил",
        "я предложил новый подход", "мы изменили процесс",
        "я рискнул", "я попробовал", "я заметил проблему",
        "я делегировал", "я доверил", "я помог им",
        "я поблагодарил", "я отметил вклад", "мы отпраздновали"
    ]
    hits = [m for m in story_markers if m in essay.lower()]
    
    print(f"\n{name}:")
    print(f"  числа: {numbers}")
    print(f"  story markers: {hits}")
    print(f"  past verbs: {past_verbs[:10]}")
    print(f"  future verbs: {future_verbs[:5]}")
    print(f"  concreteness: {detect_concreteness(essay)}")


СИЛЬНОЕ:
  числа: ['40']
  story markers: []
  past verbs: ['возглавил', 'столкнулись', 'урезали', 'покинули', 'стал', 'собрал', 'объяснил', 'предложил', 'провёл', 'убедил']
  future verbs: ['ведёт']
  concreteness: 0.4

СЛАБОЕ:
  числа: []
  story markers: []
  past verbs: ['поставленные']
  future verbs: ['поможет']
  concreteness: 0.4

ВОЗДУХ:
  числа: []
  story markers: []
  past verbs: []
  future verbs: []
  concreteness: 0.4
